# Task 1: Evolución del Espacio Latente en Stable Diffusion
### Integrantes:

* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007


## Instalación de dependencias
Instalamos las librerías que requiere el laboratorio:
- diffusers: contiene StableDiffusionPipeline, el VAE, el Scheduler y la U-Net
- transformers: provee el encoder de texto CLIP que convierte el prompt en embeddings
- accelerate: optimiza la carga y distribución del modelo en GPU
- torch: framework de tensores sobre el que corre todo el proceso

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt


## Paso 1: Instanciar el modelo y enviarlo a la GPU

StableDiffusionPipeline.from_pretrained() descarga los pesos desde Hugging Face (~4 GB la primera vez) y los cachea localmente.

- `torch_dtype=torch.float16`: carga el modelo en precisión FP16 en lugar de FP32. Reduce el uso de VRAM a la mitad con pérdida de calidad mínima — necesario para caber en la GPU T4 de Colab (15 GB VRAM).
- `.to("cuda")`: mueve todos los componentes del pipeline a la GPU: U-Net, VAE y el encoder de texto CLIP.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")
print("Modelo cargado en:", next(pipe.unet.parameters()).device)


## Paso 2: Definir 20 pasos de inferencia y fijar la semilla aleatoria

- `torch.manual_seed(42)`: fija el generador de números aleatorios. Esto garantiza que el **ruido inicial** del que parte el proceso sea siempre el mismo — sin esto, cada ejecución produce una imagen diferente.
- `num_inference_steps = 20`: el scheduler dividirá el proceso de denoising en exactamente 20 pasos, como pide el enunciado.

In [ ]:
torch.manual_seed(42)
num_inference_steps = 20
print(f"Semilla fijada. Pasos de inferencia: {num_inference_steps}")


## Paso 3: Definir el prompt

El enunciado exige este prompt específico. Fue diseñado para forzar al modelo a generar **estructuras geométricas complejas** (fruta futurista) y **texturas de alta frecuencia** (luces neón, detalles 4k), lo que hace más evidente la diferencia entre los pasos iniciales y finales del denoising.

In [ ]:
prompt = "A highly detailed cinematic and futuristic fruit glowing in a cyberpunk laboratory, neon lights, 4k resolution"
print("Prompt:", prompt)


## Pasos 4 y 5: Interceptar el tensor latente en los pasos 4, 10, 16 y 20

Este es el núcleo del Task 1. El enunciado dice que **no debemos generar la imagen de un solo golpe**, sino interceptar el tensor latente durante el loop de denoising.

### ¿Qué es el tensor latente?
Stable Diffusion no trabaja directamente en píxeles. El VAE encoder comprime la imagen a un espacio latente 8× más pequeño. Una imagen de 512×512 px tiene un tensor latente de forma `(1, 4, 64, 64)` — 4 canales de 64×64. La U-Net elimina el ruido en ese espacio comprimido paso a paso.

### Estrategia elegida: `callback_on_step_end`
El enunciado da dos opciones (hint): usar `callback_on_step_end` o escribir el loop manualmente con `scheduler.step()`. Usamos el callback porque es más limpio y no requiere reescribir el pipeline entero.

La función `callback_on_step_end` es llamada automáticamente por el pipeline **al terminar cada paso**. Recibe `step_index` (0-indexado), por lo que el paso 4 llega como índice 3, el 10 como 9, etc.

**`.clone()` es crítico:** sin él guardaríamos una referencia al mismo tensor que el pipeline sigue modificando. Al final todos los tensores guardados serían iguales al del paso 20.

In [ ]:
capture_steps = {4, 10, 16, 20}
saved_latents = {}

def callback_on_step_end(pipe, step_index, timestep, callback_kwargs):
    current_step = step_index + 1  # step_index es 0-indexado
    if current_step in capture_steps:
        saved_latents[current_step] = callback_kwargs["latents"].clone()
        print(f"  [Paso {current_step}] Tensor capturado | shape: {saved_latents[current_step].shape}")
    return callback_kwargs

print("Iniciando denoising...")
result = pipe(
    prompt,
    num_inference_steps=num_inference_steps,
    callback_on_step_end=callback_on_step_end,
    callback_on_step_end_tensor_inputs=["latents"]
)

final_image = result.images[0]
print(f"\nDenoising completo. Tensores guardados en pasos: {sorted(saved_latents.keys())}")


## Paso 6: Decodificar los 4 tensores latentes con `vae.decode()`

### ¿Por qué dividir por `scaling_factor`? (Warning del enunciado)
El pipeline de Stable Diffusion escala matemáticamente los tensores latentes antes de trabajar con ellos: multiplica cada latente por `scaling_factor` (~0.18215) para normalizar la varianza. Cuando guardamos el tensor desde el callback, lo obtenemos ya escalado.

El VAE decoder fue entrenado esperando tensores en su escala original (sin ese factor). Si le pasamos el tensor escalado directamente, el decoder recibe valores fuera de su rango esperado → imágenes completamente negras.

Solución: dividir por `scaling_factor` antes de `vae.decode()`.

In [ ]:
scaling_factor = pipe.vae.config.scaling_factor
print(f"VAE scaling_factor: {scaling_factor}")

decoded_images = {}

for step in sorted(saved_latents.keys()):
    latent = saved_latents[step]
    with torch.no_grad():
        scaled = latent / scaling_factor
        decoded = pipe.vae.decode(scaled).sample
    decoded = (decoded / 2 + 0.5).clamp(0, 1)
    decoded = decoded.cpu().permute(0, 2, 3, 1).float().numpy()
    decoded_images[step] = Image.fromarray((decoded[0] * 255).astype(np.uint8))
    print(f"Paso {step} decodificado -> tamaño: {decoded_images[step].size}")

## Paso 7: Convertir a PIL, guardar y mostrar cuadrícula

Las imágenes ya fueron convertidas a formato PIL en el paso anterior. Aquí las guardamos individualmente y mostramos la cuadrícula comparativa que pide el enunciado.

In [ ]:
for step, img in decoded_images.items():
    img.save(f"latent_step_{step}.png")
    print(f"Guardada: latent_step_{step}.png")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Evolución del Espacio Latente — Stable Diffusion v1.5", fontsize=14, fontweight="bold")

for idx, step in enumerate([4, 10, 16, 20]):
    axes[idx].imshow(decoded_images[step])
    axes[idx].set_title(f"Paso {step} / 20", fontsize=12)
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig("latent_evolution_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Cuadrícula guardada como latent_evolution_grid.png")

## Entrega: Análisis Teórico. Paso 4 vs Paso 16

* **Pregunta del enunciado:** Explicado con base en la teoría de frecuencias espaciales y Cross-Attention: ¿Qué características resuelve la U-Net en las etapas iniciales de ruido alto, y qué resuelve en las etapas finales de ruido bajo?
